### Working with Pytoch Tensors

In [1]:
import torch
import numpy as np

### Working with PyTorch tensors

In [2]:
### Default datatype float32
x = torch.ones(2,2)
print(x)
print(x.dtype)

tensor([[1., 1.],
        [1., 1.]])
torch.float32


In [3]:
### Define specific datatype
x = torch.ones(2,2, dtype=torch.int8)
print(x)
print(x.dtype)

tensor([[1, 1],
        [1, 1]], dtype=torch.int8)
torch.int8


In [4]:
### Change datatype float64, it was int8
x = x.type(torch.float64)
print(x.dtype)

torch.float64


In [5]:
### Converting tensors into Numpy arrays
x = torch.rand(2,2)
print("Tensor x is :\n",x)
print(x.dtype)

######################################
print("#"*20)
######################################

y = x.numpy()
print("Array y is:\n",y)
print(y.dtype)

Tensor x is :
 tensor([[0.5143, 0.2763],
        [0.9103, 0.1888]])
torch.float32
####################
Array y is:
 [[0.51425076 0.27626657]
 [0.91027474 0.18878698]]
float32


In [6]:
### Convert Numpy arrays into tensors
x = np.zeros((2,2), dtype=np.float32)
print("Numpy array x is:\n",x)
print(x.dtype)

######################################
print("#"*20)
######################################

y = torch.from_numpy(x)
print("Tensor y is:\n",y)
print(y.dtype)

Numpy array x is:
 [[0. 0.]
 [0. 0.]]
float32
####################
Tensor y is:
 tensor([[0., 0.],
        [0., 0.]])
torch.float32


In [7]:
### Moving tensor between devices
# Define tensor on CPU

x = torch.tensor([1.5,2])
print("Tensor is:\n",x)
print("Tensor is on device: ",x.device)

######################################
print("#"*20)
######################################

# Define Device
if torch.cuda.is_available():
    device = torch.device('cuda:0')

# Move tensor onto the CUDA device
x = x.to(device)
print("Tensor is\n",x)
print(x.device)

######################################
print("#"*20)
######################################

# Move tensor onto the CPU
device = torch.device("cpu")
x=x.to(device=device)
print("Tensor is\n",x)
print("Device is ",x.device)

######################################
print("#"*20)
######################################

# Create tensor directly on any device:
device=torch.device("cuda:0")
x=torch.ones((2,3), device=device)
print("Tensor is\n",x)
print("Tensor shape is ",x.shape)
print("Device is ",x.device)


Tensor is:
 tensor([1.5000, 2.0000])
Tensor is on device:  cpu
####################
Tensor is
 tensor([1.5000, 2.0000], device='cuda:0')
cuda:0
####################
Tensor is
 tensor([1.5000, 2.0000])
Device is  cpu
####################
Tensor is
 tensor([[1., 1., 1.],
        [1., 1., 1.]], device='cuda:0')
Tensor shape is  torch.Size([2, 3])
Device is  cuda:0


### Loading and Processing Data

In [8]:
# 1. First Load MNIST training dataset

from torchvision import datasets
path2data = "./data"
train_data = datasets.MNIST(path2data, train=True, download=True)

In [9]:
# 2. Extract the input data and target labels
x_train, y_train = train_data.data, train_data.targets
print("X-train shape: ",x_train.shape)
print("Y-train shape: ",y_train.shape)

X-train shape:  torch.Size([60000, 28, 28])
Y-train shape:  torch.Size([60000])


In [10]:
# 3. Load MNIST test dataset:
val_data = datasets.MNIST(path2data, train=False, download=True)

In [11]:
# 4. Extract data and targets
x_val, y_val = val_data.data, val_data.targets
print("X-validation shape: ",x_val.shape)
print("Y-validation shape: ", y_val.shape)

X-validation shape:  torch.Size([10000, 28, 28])
Y-validation shape:  torch.Size([10000])


In [12]:
# 5. Add new dimension to the tensors
## Pytorch requires B*C*H*W 
### Batch size(Number of images) = 60000
### Number of channels(Color Space) = 1
### Image Height = 28
### Image Width = 28

if len(x_train.shape) == 3:
    x_train = x_train.unsqueeze(1)
print("X-train shape \n",x_train.shape)

if len(x_val.shape) == 3:
    x_val = x_val.unsqueeze(1)
print("X-validation shape \n",x_val.shape)

X-train shape 
 torch.Size([60000, 1, 28, 28])
X-validation shape 
 torch.Size([10000, 1, 28, 28])


In [13]:
# 6. Import some packages
from torchvision import utils
import matplotlib.pyplot as plt
import numpy as np
%matplotlib inline

In [14]:
# 7. Define helper functions to display tensors as image:
def show(img):
    # Convert tensor to numpy array
    npimage=img.numpy()

    # Convert numpy array to H*W*C shape
    npimg_trans = np.transpose(npimage,(1,2,0))
    plt.imshow(npimg_trans, interpolation="nearest")

In [15]:
# 8. Create grid images and display them
# make grid of 40 images, 8 images per row
x_grid = utils.make_grid(x_train[:40], nrow=8, padding=2)
print(x_grid.shape)

show(x_grid)

torch.Size([3, 152, 242])


### Data Transformations (Augmentation)

#### Data Transformation

In [16]:
# 1. Define transform class to apply some image transformation
from torchvision import transforms

path2data = "./data"

#Loading MNIST training dataset
x_train = datasets.MNIST(path2data, train=True, download=True)

#Define transformation
data_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(p=1),
    transforms.RandomVerticalFlip(p=1),
    transforms.ToTensor(),
])

In [17]:
# 2. Apply transformation on an image from MNIST

# Get sample image
img = x_train[50][0]

# Transform image
img_trans = data_transform(img)

# Conver tensor to numpy array
img_trans_np = img_trans.numpy()

plt.subplot(1,2,1)
plt.imshow(img,cmap="gray")
plt.title("original")
plt.subplot(1,2,2)
plt.imshow(img_trans_np[0],cmap="gray");
plt.title("transformed")


Text(0.5, 1.0, 'transformed')

In [18]:
# 3. Pass the transformet function to the dataset class:
data_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(p=1),
    transforms.RandomVerticalFlip(p=1),
    transforms.ToTensor(),
])

# Loading MNIST training data with on-the-fly transformations
train_data = datasets.MNIST(path2data, train=True, download=True, transform=data_transform)

#### Wrapping tensors into Dataset and DataLoader

In [46]:
# 1. Create Pytorch dataset by wrapping x_train and y_train
## Wrap tensors into dataset

from torch.utils.data import TensorDataset

train_ds = TensorDataset(x_train, y_train)
val_ds = TensorDataset(x_val, y_val)

for x,y in train_ds:
    print(x.shape, y.item())
    break

torch.Size([28, 28]) 5


#### Creating Data Loaders

In [47]:
# 1. Create 2 data loaders for traing and validation
from torch.utils.data import DataLoader

# Create a data loader from dataset
train_dl = DataLoader(train_ds, batch_size=8)
val_dl = DataLoader(val_ds,batch_size=8)

# Iterate over batches
for xb, yb in train_dl:
    print(xb.shape)
    print(yb.shape)
    break

torch.Size([8, 28, 28])
torch.Size([8])


### Building Models

#### Defining a Linear Layer

In [62]:
from torch import nn 

# Input Tensor Dimension 64*1000
input_tensor = torch.randn(64,1000)

# Linear Layer with 1000 inputs and 100 outputs
linear_layer = nn.Linear(1000,10)

# Output of Linear Layer
output = linear_layer(input_tensor)

print(output.shape)

torch.Size([64, 10])


In [65]:
model = nn.Sequential(
    nn.Linear(4,5),
    nn.ReLU(),
    nn.Linear(5,1),
)
print(model)

Sequential(
  (0): Linear(in_features=4, out_features=5, bias=True)
  (1): ReLU()
  (2): Linear(in_features=5, out_features=1, bias=True)
)


### Defining Models Using nn.Module


In [66]:
# Defining Network Class 
import torch.nn.functional as F

class Net(nn.Module):
    def __init__(self):
        super(Net,self).__init__()

        self.conv1 = nn.Conv2d(1,20,5,1)
        self.conv2 = nn.Conv2d(20,50,5,1)
        self.fc1 = nn.Linear(4*4*50,500)
        self.fc2 = nn.Linear(500,10)
    
    def forward(self,x):
        x = F.relu(self.con1(x))
        x = F.max_pool2d(x,2,2)
        x = F.relu(self.conv2(x))
        x = F.max_pool2d(x,2,2)
        x = x.view(-1,4*4*50)
        x = F.relu(self.fc1(x))
        x = self.fc2(x) 

        return F.log_softmax(x, dim=1)

model = Net()
print(model)


Net(
  (conv1): Conv2d(1, 20, kernel_size=(5, 5), stride=(1, 1))
  (conv2): Conv2d(20, 50, kernel_size=(5, 5), stride=(1, 1))
  (fc1): Linear(in_features=800, out_features=500, bias=True)
  (fc2): Linear(in_features=500, out_features=10, bias=True)
)
